In [1]:
%pip install scikit-learn xgboost
%pip install imblearn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.svm import SVC
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import numpy as np
random_seed = 2026
np.random.seed(random_seed)

In [2]:
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

## Load Data

In [8]:
def load_data(path, channels=["delta", "theta", "alpha", "beta","gamma"]):
    # Load data
    data = pd.read_pickle(path).reset_index(drop=True)
    
    # Split data
    groups = data["subject"]
    X = data[[column for column in data.columns if any(channel in column for channel in channels)]].astype(float)
    y = data["is_mw"].astype(int)
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=random_seed)
    return X, y, groups, cv

## Training
### Logistic Regression

In [3]:
def train_logistic_regression(X, y, groups, parameters):
    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_seed)

    lr = LogisticRegression(random_state=random_seed)
    clf = GridSearchCV(lr, parameters, cv=cv, scoring="roc_auc") 

    clf.fit(X,y,groups=groups)
    
    return clf

### SVM with radial basis function

In [4]:
def train_svc(X, y, groups, parameters):
    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_seed)

    svc = SVC(probability = True,random_state=random_seed)
    clf = GridSearchCV(svc, parameters, cv=cv, scoring="roc_auc") 

    clf.fit(X,y,groups=groups)
    
    return clf

### XGBoost

In [5]:
def train_xgboost(X, y, groups, parameters):
    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_seed)

    xgb = XGBClassifier(scale_pos_weight=len(y_trainval[y_trainval==0])/len(y_trainval[y_trainval==1]), random_state=random_seed, eval_metric="auc")
    clf = GridSearchCV(xgb, parameters, cv=cv, scoring="roc_auc") 

    clf.fit(X,y,groups=groups)
    
    return clf

### Training

In [6]:
lr_params = { # Why these?
    'C': [0.01, 0.1, 1],#, 10, 100], 
    'class_weight': ["balanced"],
    'solver': ["lbfgs"],
    'max_iter': [1000],
    'warm_start': [True]#, False]
}

svm_params = { # Why these?
    'C': [0.01,0.1,1],#, 10, 100],
    'gamma': ["auto"]#,"scale", 0.01, 0.1]
}

xgb_params = { # Why these?
    'learning_rate': [0.01],#, 0.1],
    'n_estimators': [100,200,300],#, 200],
    'max_depth': [5,10,15],#, 10],
    'reg_alpha': [1,3],#, 5]
}

In [9]:
path = "data/preprocessed/data_channels_2000ms.pkl"
X, y, groups, cv = load_data(path) # This loads all columns. You can filter for specific columns by adding filter substrings to the 'channels' parameter list.
X.head()

,delta_Fp1,delta_AF7,delta_AF3,delta_F1,delta_F3,delta_F5,delta_F7,delta_FT7,delta_FC5,delta_FC3,...,gamma_CP4,gamma_CP2,gamma_P2,gamma_P4,gamma_P6,gamma_P8,gamma_P10,gamma_PO8,gamma_PO4,gamma_O2
0,2.462158,-0.786966,1.453841,-0.173519,-0.769492,-0.287671,-0.016817,0.106244,-0.263572,-0.469618,...,1.657072,0.698362,0.887055,1.705181,1.210251,7.597670,6.660648,4.159779,1.046496,0.598613
1,-0.763733,-0.763764,-0.839963,-0.751324,-0.641978,-0.288063,-0.079681,-0.898491,-0.984690,-0.238345,...,0.712813,1.334872,0.824546,1.228759,0.946114,4.261911,2.271643,1.681627,0.584873,-0.070074
2,1.837950,0.279270,2.548398,3.484596,3.426255,0.389319,-0.071008,-0.581717,-0.112547,1.432430,...,2.790258,1.954135,0.441275,1.204265,0.282846,0.065358,0.035225,-0.480892,-0.210074,-0.642990
3,1.021164,-0.041133,0.208394,0.182622,0.679224,0.241317,0.008427,-0.222973,-0.396165,0.456203,...,0.522042,0.611324,0.837795,0.429085,0.173270,1.052851,0.386238,0.139689,0.180052,-0.365811
4,0.363276,-0.464267,-0.007265,0.148546,0.289437,-0.376768,-0.103516,-0.918463,-0.986162,-0.488684,...,0.236916,0.645471,0.714078,0.989184,-0.021414,1.342126,0.248375,0.056425,0.155765,-0.487687


In [ ]:


# for fold, (train_idxs, test_idxs) in enumerate(cv.split(X, y, groups)):
#     X_trainval, y_trainval = X.iloc[train_idxs].reset_index(drop=True), y[train_idxs].reset_index(drop=True)
#     X_test, y_test = X.iloc[test_idxs].reset_index(drop=True), y[test_idxs].reset_index(drop=True)
#     groups_train = groups[train_idxs]
#     groups_test = groups[test_idxs]

#     print(f"Fold {fold}")
#     # best_logreg_clf = train_logistic_regression(X_trainval, y_trainval, groups_train, parameters=lr_params)
#     # perf_auc = roc_auc_score(y_test, best_logreg_clf.predict_proba(X_test)[:,1])
#     # perf_f1 = f1_score(y_test, best_logreg_clf.predict(X_test))
#     # print(f"LogisticRegression({best_logreg_clf.best_params_}): AUC={perf_auc}; F1={perf_f1}")

#     best_svm_clf = train_svc(X_trainval, y_trainval, groups_train, parameters=svm_params)
#     perf_auc = roc_auc_score(y_test, best_svm_clf.decision_function(X_test))
#     perf_f1 = f1_score(y_test, best_svm_clf.predict(X_test))
#     print(f"SVC({best_svm_clf.best_params_}): AUC={perf_auc}; F1={perf_f1}")

#     # best_xgb_clf = train_xgboost(X_trainval, y_trainval, groups_train, parameters=xgb_params)
#     # perf_auc = roc_auc_score(y_test, best_xgb_clf.predict_proba(X_test)[:,1])
#     # perf_f1 = f1_score(y_test, best_xgb_clf.predict(X_test))
#     # print(f"XGBoost({best_xgb_clf.best_params_}): AUC={perf_auc}; F1={perf_f1}")
#     # print()
#     # print("Fold :", fold)
#     # print("TRAIN POSITIVE RATIO:", y[train_idxs].mean())
#     # print("TEST POSITIVE RATIO :", y[test_idxs].mean())
#     # print(f"TRAIN PERCENTAGE    : {len(train_idxs) / len(X) * 100:.4}%")
#     # print("TRAIN GROUPS        :", set(groups[train_idxs]))
#     # print("TEST GROUPS         :", set(groups[test_idxs]))


Fold 0


/home/stefan/Documents/GitHub/BCI-Mind-Wandering/.conda/lib/python3.11/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/stefan/Documents/GitHub/BCI-Mind-Wandering/.conda/lib/python3.11/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/stefan/Documents/GitHub/BCI-Mind-Wandering/.conda/lib/python3.11/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/stefan/Documents/GitHub/BCI-Mind-Wandering/

KeyboardInterrupt: 

In [ ]:
#with smote
logreg_aucs = []
logreg_best_params = None
svc_aucs = []
svc_best_params = None
xgb_aucs = []
xgb_best_params = None

for fold, (train_idxs, test_idxs) in enumerate(cv.split(X, y, groups)):
    X_trainval, y_trainval = X.iloc[train_idxs].reset_index(drop=True), y[train_idxs].reset_index(drop=True)
    X_test, y_test = X.iloc[test_idxs].reset_index(drop=True), y[test_idxs].reset_index(drop=True)
    smote = SMOTE(sampling_strategy='minority', random_state=random_seed)
    X_trainval_sm, y_trainval_sm = smote.fit_resample(X_trainval, y_trainval)
    groups_train = groups[train_idxs]
    groups_test = groups[test_idxs]
    rest = len(X_trainval_sm) - len(X_trainval)
    groups_train = pd.concat([groups_train, pd.Series(np.random.choice(groups_train.unique(), size = rest))])

    print(f"Fold {fold}")

    # Logistic Regression
    best_logreg_clf = train_logistic_regression(X_trainval_sm, y_trainval_sm, groups_train, parameters=lr_params)
    perf_auc = roc_auc_score(y_test, best_logreg_clf.predict_proba(X_test)[:,1])
    logreg_aucs.append(perf_auc)
    if perf_auc == max(logreg_aucs):
        logreg_best_params = best_logreg_clf.best_params_
    perf_f1 = f1_score(y_test, best_logreg_clf.predict(X_test))
    print(f"LogisticRegression({best_logreg_clf.best_params_}): AUC={perf_auc}; F1={perf_f1}")

    # Support Vector Classifier
    best_svm_clf = train_svc(X_trainval_sm, y_trainval_sm, groups_train, parameters=svm_params)
    perf_auc = roc_auc_score(y_test, best_svm_clf.decision_function(X_test))
    svc_aucs.append(perf_auc)
    if perf_auc == max(svc_aucs):
        svc_best_params = best_svm_clf.best_params_
    perf_f1 = f1_score(y_test, best_svm_clf.predict(X_test))
    print(f"SVC({best_svm_clf.best_params_}): AUC={perf_auc}; F1={perf_f1}")

    # XGBoost Classifier
    best_xgb_clf = train_xgboost(X_trainval_sm, y_trainval_sm, groups_train, parameters=xgb_params)
    perf_auc = roc_auc_score(y_test, best_xgb_clf.predict_proba(X_test)[:,1])
    xgb_aucs.append(perf_auc)
    if perf_auc == max(xgb_aucs):
        xgb_best_params = best_xgb_clf.best_params_
    perf_f1 = f1_score(y_test, best_xgb_clf.predict(X_test))
    print(f"XGBoost({best_xgb_clf.best_params_}): AUC={perf_auc}; F1={perf_f1}")
    
    print()
    # print("Fold :", fold)
    # print("TRAIN POSITIVE RATIO:", y[train_idxs].mean())
    # print("TEST POSITIVE RATIO :", y[test_idxs].mean())
    # print(f"TRAIN PERCENTAGE    : {len(train_idxs) / len(X) * 100:.4}%")
    # print("TRAIN GROUPS        :", set(groups[train_idxs]))
    # print("TEST GROUPS         :", set(groups[test_idxs]))


Fold 0
LogisticRegression({'C': 1, 'class_weight': 'balanced', 'max_iter': 1000, 'solver': 'lbfgs', 'warm_start': True}): AUC=0.5761619789510173; F1=0.09933035714285714


In [ ]:
print(f"Mean Logistic Regression  AUC = {sum(logreg_aucs)/len(logreg_aucs)} ({logreg_best_params})") 
print(f"Mean SVC AUC = {sum(svc_aucs)/len(svc_aucs)} ({svc_best_params})")
print(f"Mean XGBoost AUC = {sum(xgb_aucs)/len(xgb_aucs)} ({xgb_best_params})")

Mean Logistic Regression  AUC = 0.5546344190655784 ({'C': 1, 'class_weight': 'balanced', 'max_iter': 1000, 'solver': 'lbfgs', 'warm_start': True})
